# Model Comparison — Diabetes 30-Day Readmission

**Project:** DS4400 — 30-Day Readmission Prediction in Diabetic Patients  
**Team:** Om Prajapati, Nafisa Tasnia, Sraghvi Anchaliya

Here we retrain each model with the best hyperparameters we found in the individual notebooks and compare them on the same test set.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('imports done')

ModuleNotFoundError: No module named 'xgboost'

## 1. Load Data

In [ ]:
X_train_full = pd.read_csv('../Data/X_train.csv')
X_test_full  = pd.read_csv('../Data/X_test.csv')
y_train      = pd.read_csv('../Data/y_train.csv').squeeze()
y_test       = pd.read_csv('../Data/y_test.csv').squeeze()

top_features = pd.read_csv('../Data/top_features.csv', header=None).squeeze().tolist()

X_train_top = X_train_full[top_features].copy()
X_test_top  = X_test_full[top_features].copy()

# add interaction features (used by LR, tested by all models)
interactions = {
    'inpatient_x_medications': ('number_inpatient', 'num_medications'),
    'inpatient_x_time':        ('number_inpatient', 'time_in_hospital'),
    'age_x_diagnoses':         ('age',              'number_diagnoses'),
    'age_x_inpatient':         ('age',              'number_inpatient'),
    'insulin_x_inpatient':     ('insulin',          'number_inpatient'),
}

for name, (a, b) in interactions.items():
    X_train_top[name] = X_train_top[a] * X_train_top[b]
    X_test_top[name]  = X_test_top[a]  * X_test_top[b]

# SVM needs a subsample — too slow on full training set
SUBSAMPLE_SIZE = 20_000
X_sub, _, y_sub, _ = train_test_split(
    X_train_top, y_train, train_size=SUBSAMPLE_SIZE, random_state=42, stratify=y_train
)

# XGBoost uses its own feature selection (44 features via SelectFromModel)
# but for fair comparison we'll use the same top features + interactions here
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'Top features + interactions: {X_train_top.shape[1]}')
print(f'Train: {X_train_top.shape[0]}  |  Test: {X_test_top.shape[0]}')
print(f'SVM subsample: {X_sub.shape[0]}')
print(f'Positive rate: {y_train.mean():.3f}')

## 2. Train All Models (Best Hyperparameters)

Retrain each model with the best params from their individual notebooks so we can compare them fairly on the same features and test split.

In [ ]:
# ---- Logistic Regression (best from LR notebook: class_weight='balanced') ----
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_top, y_train)
print('LR trained')

# ---- SVM (best from SVM notebook: C=1, gamma=0.01, on subsample) ----
svm = SVC(kernel='rbf', C=1, gamma=0.01, class_weight='balanced',
          random_state=42, probability=True)
svm.fit(X_sub, y_sub)
print('SVM trained (20k subsample)')

# ---- Random Forest (best from RF notebook: 200 trees, max_depth=10) ----
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=2,
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_top, y_train)
print('RF trained')

# ---- XGBoost (best from XGB notebook) ----
xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.7, min_child_weight=3, gamma=0,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train_top, y_train)
print('XGBoost trained')

## 3. Collect Predictions & Metrics

In [ ]:
models = {
    'Logistic Regression': lr,
    'SVM (RBF)':           svm,
    'Random Forest':       rf,
    'XGBoost':             xgb,
}

results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test_top)
    y_prob = model.predict_proba(X_test_top)[:, 1]

    predictions[name] = {'y_pred': y_pred, 'y_prob': y_prob}

    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1':        round(f1_score(y_test, y_pred), 4),
        'Macro F1':  round(f1_score(y_test, y_pred, average='macro'), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())

## 4. Metrics Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
results_df[['F1', 'Recall', 'Precision', 'ROC-AUC']].plot(
    kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7
)
ax.set_title('Model Comparison — Key Metrics (Minority Class)', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(loc='upper right')

# add value labels on each bar
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('../Results/Comparison_Results/model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. ROC Curves (All Models)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
for name, preds in predictions.items():
    RocCurveDisplay.from_predictions(y_test, preds['y_prob'], ax=axes[0], name=name)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random baseline')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(fontsize=9)

# Precision-Recall curves
for name, preds in predictions.items():
    PrecisionRecallDisplay.from_predictions(y_test, preds['y_prob'], ax=axes[1], name=name)
baseline_rate = y_test.mean()
axes[1].axhline(baseline_rate, color='k', linestyle='--', label=f'Random ({baseline_rate:.2f})')
axes[1].set_title('Precision-Recall Curves — All Models')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../Results/Comparison_Results/model_comparison_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Confusion Matrices (Side by Side)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No', '<30d'], yticklabels=['No', '<30d'])
    f1_val = f1_score(y_test, preds['y_pred'])
    ax.set_title(f'{name}\nF1={f1_val:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../Results/Comparison_Results/model_comparison_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Discussion

### Why are all F1 scores so low?

Honestly this confused us at first but after looking into it, this is just a really hard dataset. A few reasons:

1. Only ~9% of patients are readmitted within 30 days, so even with class weighting the models struggle to pick up on the minority class without also flagging a ton of false positives.

2. The features we have (demographics, medications, lab results) just don't capture a lot of what actually determines if someone gets readmitted. Things like whether the patient follows up with their doctor or has support at home aren't in the data at all.

3. Other papers using this exact dataset get similar numbers — F1 around 0.22-0.35 for the minority class and AUC around 0.63-0.70. So we're right in that range.

### What we noticed about each model

Logistic regression is the simplest model but it actually does pretty well here. The coefficients are easy to interpret which would matter in a clinical setting. The interaction features (like number_inpatient * num_medications) gave it a small boost since LR can't learn those patterns on its own.

SVM does about the same as LR but takes way longer to train and you can't really interpret it. The RBF kernel should be able to find nonlinear patterns but there doesn't seem to be much nonlinear structure in this data that LR is missing.

Random Forest is good at telling us which features matter (feature importance) and limiting the tree depth to 10 was important — without it the trees just memorize the training data.

XGBoost gets the best ROC-AUC since boosting focuses on the hard-to-classify examples, but the minority class F1 ends up similar to the other models.

### Precision vs recall tradeoff

Every model has the same problem: if you want to catch more readmissions (higher recall) you also end up flagging a lot of patients who weren't actually going to be readmitted (lower precision). In a hospital you'd have to decide what's worse — missing someone who needs follow-up, or wasting resources on false alarms.

### Features that matter

All four models agree on the important features: `number_inpatient` (prior hospital visits) is #1 everywhere, followed by `time_in_hospital`, discharge disposition, and `num_medications`/`number_diagnoses` which are basically proxies for how sick someone is.